<a href="https://colab.research.google.com/github/tranlongnhat-git/License-Plate-Recognition-System/blob/main/OCR_MAIN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

import os
import yaml
import shutil
import datetime
from google.colab import drive

MODEL_VERSION = 'n'

WEIGHTS = f'yolov5{MODEL_VERSION}.pt'
BATCH_SIZE = 64
EPOCHS = 150

# --- BƯỚC 1: KẾT NỐI DRIVE & TÌM DỮ LIỆU ---
print("🚀 Đang khởi động quy trình...")
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive', force_remount=True)

# Tự động tìm thư mục OCR
possible_paths = [
    '/content/drive/MyDrive/OCR'
]
DRIVE_OCR_PATH = None
for path in possible_paths:
    if os.path.exists(path):
        DRIVE_OCR_PATH = path
        print(f"✅ Đã tìm thấy dữ liệu gốc tại: {DRIVE_OCR_PATH}")
        break

if not DRIVE_OCR_PATH:
    raise FileNotFoundError("❌ LỖI: Không tìm thấy thư mục 'OCR' trên Drive của bạn!")

# Tạo nơi lưu model kết quả
FINAL_MODEL_DIR = os.path.join(os.path.dirname(DRIVE_OCR_PATH), 'My_Models_Lightweight')
if not os.path.exists(FINAL_MODEL_DIR): os.makedirs(FINAL_MODEL_DIR)

# --- BƯỚC 2: COPY DỮ LIỆU VÀO MÁY ẢO
LOCAL_DATA_DIR = '/content/temp_data/OCR'
if not os.path.exists(LOCAL_DATA_DIR):
    print(f"⏳ Đang copy dữ liệu xuống máy ảo để train nhanh (chờ xíu)...")
    shutil.copytree(DRIVE_OCR_PATH, LOCAL_DATA_DIR)
    print("✅ Copy dữ liệu hoàn tất.")
else:
    print("✅ Dữ liệu máy ảo đã sẵn sàng.")

# --- BƯỚC 3: CÀI ĐẶT YOLOv5 MỚI NHẤT ---
if not os.path.exists('/content/yolov5'):
    print("⬇️ Đang tải và cài đặt YOLOv5...")
    !git clone https://github.com/ultralytics/yolov5 /content/yolov5
    %cd /content/yolov5
    !pip install -r requirements.txt -q
    !wandb disabled 2>/dev/null
else:
    %cd /content/yolov5
    print("✅ Môi trường YOLOv5 đã sẵn sàng.")

# --- BƯỚC 4: TẠO FILE CẤU HÌNH  ---
# ID 0-8: Số 1 đến 9
# ID 9-28: Chữ cái (A-Z, bỏ I,J,O,Q,R,W)
# ID 29: Số 0
correct_names = [
    '1', '2', '3', '4', '5', '6', '7', '8', '9',
    'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'K', 'L',
    'M', 'N', 'P', 'S', 'T', 'U', 'V', 'X', 'Y', 'Z',
    '0'
]

ocr_yaml = {
    'train': f'{LOCAL_DATA_DIR}/images/train',
    'val': f'{LOCAL_DATA_DIR}/images/val',
    'nc': 30,
    'names': correct_names
}
with open('/content/ocr_config.yaml', 'w') as f: yaml.dump(ocr_yaml, f)
print(f"✅ Đã tạo cấu hình với {len(correct_names)} class chuẩn.")

# --- BƯỚC 5: TRAIN ---
print("\n" + "="*50)
print(f"🔥 BẮT ĐẦU TRAIN PHIÊN BẢN: YOLOv5-{MODEL_VERSION.upper()} (OCR)")
print("="*50)

!python train.py --img 640 \
                 --batch {BATCH_SIZE} \
                 --epochs {EPOCHS} \
                 --data /content/ocr_config.yaml \
                 --weights {WEIGHTS} \
                 --project /content/runs/train \
                 --name ocr_light_exp \
                 --exist-ok

# --- BƯỚC 6: LƯU TOÀN BỘ KẾT QUẢ (UPDATED) ---
print("\n💾 Đang lưu TOÀN BỘ kết quả (Model + Biểu đồ + Log) về Drive...")

# Định nghĩa nguồn và đích
source_dir = '/content/runs/train/ocr_light_exp'
timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
# Tạo tên thư mục riêng biệt theo thời gian để không bị ghi đè
destination_dir = os.path.join(FINAL_MODEL_DIR, f'OCR_Result_{timestamp}_{MODEL_VERSION.upper()}')

try:
    if os.path.exists(source_dir):
        shutil.copytree(source_dir, destination_dir)
        print(f"🎉 THÀNH CÔNG RỰC RỠ! Đã lưu trọn bộ kết quả tại:")
        print(f"   📂 {destination_dir}")
        print("   (Hãy vào Drive kiểm tra folder này để xem biểu đồ và file weights/best.pt)")
    else:
        print("❌ Lỗi: Không tìm thấy thư mục kết quả trong Colab. Có thể quá trình train bị lỗi.")
except Exception as e:
    print(f"❌ Có lỗi khi copy sang Drive: {e}")

Kết quả truyền trực tuyến bị cắt bớt đến 5000 dòng cuối.
    100/149      10.1G    0.02079     0.0494   0.004991        959        640:  98% 47/48 [00:59<00:01,  1.36s/it]/content/yolov5/train.py:414: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(amp):
    100/149      10.1G    0.02077     0.0494   0.004987        895        640: 100% 48/48 [01:00<00:00,  1.26s/it]
                 Class     Images  Instances          P          R      mAP50   mAP50-95: 100% 6/6 [00:13<00:00,  2.22s/it]
                   all        767       6236      0.975      0.964      0.981      0.759

      Epoch    GPU_mem   box_loss   obj_loss   cls_loss  Instances       Size
  0% 0/48 [00:00<?, ?it/s]/content/yolov5/train.py:414: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(amp):
    101/149     